# 2.4 bis — Méthode apprise, APP-v2 : mieux entraîner BioBERT

**Pourquoi.** APP-v1 n'a fait que 140 étapes d'entraînement (560 fenêtres, lots de 8, 2 époques) et son F1 montait
encore fortement d'une époque à l'autre (0,74 → 0,85). Le « 2 époques » retenu était en plus le point intermédiaire
d'un entraînement prévu pour 3. Hypothèse : le modèle est sous-entraîné, surtout pour Disease (F1 0,78).

**Ce qui change par rapport à APP-v1** (même modèle, mêmes données, même découpage, même décodage) :
- jusqu'à **10 époques**, score de VALIDATION après chacune ;
- **montée progressive** du taux d'apprentissage sur les 10 % premières étapes (*warmup*), puis décroissance linéaire ;
- deux taux d'apprentissage : **3e-5** et **5e-5** ;
- deux **graines** (42 et 43) : on compare les moyennes, pour ne pas confondre un vrai gain et un tirage favorable ;
- perte non pondérée (poids 1, retenu en APP-v1).

**Règle de choix, fixée avant de lancer** : meilleure moyenne de F1 VALIDATION sur les deux graines ; parmi les
configurations à moins de 0,005 de la meilleure, celle qui a le moins d'époques. Le modèle figé est celui de la
graine 42 à cette configuration.

**Exécution** : Colab, GPU T4, environ 35 minutes. APP-v1 n'est pas modifié : tous les fichiers produits ici
commencent par `apprise_v2_` / `predictions_apprise_v2_`, le modèle va dans `modeles/biobert_v2`.

In [1]:
import os, sys, json, time, shutil, math
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RACINE = Path("/content/drive/MyDrive/projet_nlp_sante")
    DOSSIER_CHECKPOINTS = Path("/content/checkpoints")   # disque local de Colab, pas le Drive
else:
    RACINE = Path.cwd().parent
    DOSSIER_CHECKPOINTS = RACINE / "modeles" / "checkpoints"
sys.path.insert(0, str(RACINE))

# Mode rapide : vérifie le parcours sur quelques documents (CPU), sans viser une performance.
MODE_RAPIDE = os.environ.get("PROJET_MODE_RAPIDE") == "1"
print("Colab :", EN_COLAB, "| racine :", RACINE, "| mode rapide :", MODE_RAPIDE)

Mounted at /content/drive
Colab : True | racine : /content/drive/MyDrive/projet_nlp_sante | mode rapide : False


In [2]:
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (BertTokenizer, AutoModelForTokenClassification, BertForTokenClassification,
                          TrainingArguments, DataCollatorForTokenClassification, set_seed)

from tals import erreurs
from src.cdr import charger_jsonl, mentions_reference
from src.apprise import (LABELS, LABEL2ID, POLITIQUE_DECODAGE, construire_fenetres, encoder, predire,
                         WeightedLossTrainer)
from src.extraction import tableau_scores, ecrire_predictions, lire_predictions

VERSION = "APP-v2"
MODELE = "dmis-lab/biobert-base-cased-v1.1"
PARAMS = {"batch_size": 8, "weight_decay": 0.01, "max_length": 512, "entity_weight": 1.0,
          "epochs": 10, "warmup": 0.1}
TAUX_COMPARES = [3e-5, 5e-5]
GRAINES = [42, 43]
GRAINE_FIGEE = 42
TOLERANCE = 0.005   # écart de F1 en dessous duquel deux configurations sont considérées à égalité

RESULTATS = RACINE / "resultats" / "extraction"
SORTIE = RESULTATS / "essai_rapide" if MODE_RAPIDE else RESULTATS
SORTIE.mkdir(parents=True, exist_ok=True)
if MODE_RAPIDE:
    PARAMS.update(epochs=2, max_length=128)
    TAUX_COMPARES, GRAINES = [5e-5], [42]

appareil = "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil :", appareil, "| torch", torch.__version__, "| transformers", transformers.__version__)

Appareil : GPU Tesla T4 | torch 2.11.0+cu130 | transformers 5.20.2


## Étape 1 — Préparer les données

Identique à APP-v1 (2.4) : étiquettes BIO du 2.2, fenêtres de phrases entières, premier sous-mot supervisé.

In [3]:
documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")
bio = [json.loads(l) for l in open(RACINE / "data" / "bio_extraction.jsonl", encoding="utf-8")]
textes = {d.document_id: d.texte for d in documents}
bio_par_doc = {b["document_id"]: b for b in bio}
parts = {p: [d for d in documents if d.partition == p] for p in ["train", "validation", "test"]}
if MODE_RAPIDE:
    parts = {"train": parts["train"][:20], "validation": parts["validation"][:10], "test": parts["test"][:10]}

tokenizer = BertTokenizer.from_pretrained(MODELE)
fenetres = {p: construire_fenetres([bio_par_doc[d.document_id] for d in docs], textes, tokenizer, PARAMS["max_length"])
            for p, docs in parts.items()}
donnees = {p: encoder(fenetres[p], bio_par_doc, tokenizer, PARAMS["max_length"])[0] for p in ["train", "validation"]}
etapes_par_epoque = math.ceil(len(donnees["train"]) / PARAMS["batch_size"])
print({p: len(f) for p, f in fenetres.items()}, "| étapes par époque :", etapes_par_epoque,
      "| étapes au total :", etapes_par_epoque * PARAMS["epochs"])

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

{'train': 560, 'validation': 552, 'test': 563} | étapes par époque : 70 | étapes au total : 700


## Étape 2 — Entraîner et mesurer chaque époque sur VALIDATION

Un entraînement par couple (taux d'apprentissage, graine). Le modèle est sauvegardé à la fin de chaque époque puis
évalué sur VALIDATION. On ne garde sur le disque que les points de la graine 42 (seuls candidats au gel).

In [ ]:
ref_val = mentions_reference(parts["validation"])
collator = DataCollatorForTokenClassification(tokenizer)

def entrainer(taux, graine):
    set_seed(graine)
    modele = BertForTokenClassification.from_pretrained(
        MODELE, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
    dossier = DOSSIER_CHECKPOINTS / f"v2_taux{taux:g}_graine{graine}"
    args = TrainingArguments(
        output_dir=str(dossier), save_strategy="epoch", save_only_model=True, eval_strategy="no",
        logging_strategy="epoch", learning_rate=taux, per_device_train_batch_size=PARAMS["batch_size"],
        num_train_epochs=PARAMS["epochs"], weight_decay=PARAMS["weight_decay"], lr_scheduler_type="linear",
        warmup_steps=int(PARAMS["warmup"] * etapes_par_epoque * PARAMS["epochs"]),
        fp16=torch.cuda.is_available(), report_to="none", seed=graine, data_seed=graine, save_total_limit=None)
    trainer = WeightedLossTrainer(model=modele, args=args, entity_weight=PARAMS["entity_weight"],
                                  train_dataset=donnees["train"], processing_class=tokenizer, data_collator=collator)
    t0 = time.perf_counter()
    trainer.train()
    duree = time.perf_counter() - t0
    points = sorted(dossier.glob("checkpoint-*"), key=lambda c: int(c.name.split("-")[1]))
    return dossier, points, duree, trainer.state.log_history

reglage, historiques = [], {}
for taux in TAUX_COMPARES:
    for graine in GRAINES:
        dossier, points, duree, historique = entrainer(taux, graine)
        historiques[f"taux{taux:g}_graine{graine}"] = historique
        for epoque, point in enumerate(points, start=1):
            modele = AutoModelForTokenClassification.from_pretrained(point).to(DEVICE)
            pred = predire(modele, tokenizer, fenetres["validation"], bio_par_doc, PARAMS["max_length"])
            s = tableau_scores(ref_val, pred, "apprise", "validation").set_index("categorie")
            reglage.append({"taux": taux, "graine": graine, "epoques": epoque, "checkpoint": str(point),
                            "P": s.loc["global (micro)", "precision"], "R": s.loc["global (micro)", "rappel"],
                            "F1": s.loc["global (micro)", "F1"], "F1 Chemical": s.loc["Chemical", "F1"],
                            "F1 Disease": s.loc["Disease", "F1"], "temps_entrainement_s": round(duree, 1)})
            del modele
        if graine != GRAINE_FIGEE:
            shutil.rmtree(dossier)
        print(f"taux {taux:g}, graine {graine} : entraînement {duree / 60:.1f} min, "
              f"meilleur F1 {max(r['F1'] for r in reglage if r['taux'] == taux and r['graine'] == graine):.4f}")

reglage = pd.DataFrame(reglage)
reglage.drop(columns="checkpoint").to_csv(SORTIE / "apprise_v2_reglage_validation.csv", index=False)
json.dump(historiques, open(SORTIE / "apprise_v2_historique_entrainement.json", "w"), indent=2, default=str)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

... (91 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

taux 5e-05, graine 43 : entraînement 4.1 min, meilleur F1 0.8968


In [ ]:
# F1 VALIDATION par époque : moyenne des graines (et écart entre graines).
moyennes = (reglage.groupby(["taux", "epoques"])[["P", "R", "F1", "F1 Chemical", "F1 Disease"]].mean().round(4)
            .join(reglage.groupby(["taux", "epoques"])["F1"].agg(lambda f: f.max() - f.min()).round(4)
                  .rename("écart entre graines")))
moyennes["F1"].unstack("taux")

taux,0.00003,0.00005
epoques,,
1,0.7468,0.8346
2,0.8665,0.8792
3,0.8779,0.8826
4,0.8810,0.8920
5,0.8844,0.8919
6,0.8822,0.8900
7,0.8872,0.8934
8,0.8849,0.8952
9,0.8889,0.8954


## Étape 3 — Choisir et figer APP-v2

Règle annoncée en tête : meilleure moyenne ; à moins de 0,005 de la meilleure, le moins d'époques.

In [ ]:
meilleur_f1 = moyennes["F1"].max()
candidates = moyennes[moyennes["F1"] >= meilleur_f1 - TOLERANCE].reset_index()
choix = candidates.sort_values(["epoques", "F1"], ascending=[True, False]).iloc[0]
point = reglage[(reglage.taux == choix.taux) & (reglage.epoques == choix.epoques) & (reglage.graine == GRAINE_FIGEE)].iloc[0]
print(f"Configuration retenue : taux {choix.taux:g}, {int(choix.epoques)} époques — F1 moyen {choix.F1} "
      f"(meilleur moyen : {meilleur_f1}) ; modèle figé : graine {GRAINE_FIGEE}")

DOSSIER_MODELE = RACINE / "modeles" / ("biobert_v2_essai_rapide" if MODE_RAPIDE else "biobert_v2")
if DOSSIER_MODELE.exists():
    shutil.rmtree(DOSSIER_MODELE)
shutil.copytree(point.checkpoint, DOSSIER_MODELE)
modele = AutoModelForTokenClassification.from_pretrained(DOSSIER_MODELE).to(DEVICE)

t0 = time.perf_counter()
pred_val = predire(modele, tokenizer, fenetres["validation"], bio_par_doc, PARAMS["max_length"])
t_val = time.perf_counter() - t0
scores_val = tableau_scores(ref_val, pred_val, "apprise_v2", "validation", t_val)
scores_val.to_csv(SORTIE / "scores_apprise_v2_validation.csv", index=False)
ecrire_predictions(pred_val, "apprise_v2", parts["validation"], SORTIE / "predictions_apprise_v2_validation.jsonl")
scores_val

Configuration retenue : taux 5e-05, 4 époques — F1 moyen 0.892 (meilleur moyen : 0.8956) ; modèle figé : graine 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,apprise_v2,validation,Chemical,5018,360,329,0.9331,0.9385,0.9358,False,NaN
1,apprise_v2,validation,Disease,3588,719,656,0.8331,0.8454,0.8392,False,NaN
2,apprise_v2,validation,global (micro),8606,1079,985,0.8886,0.8973,0.8929,False,18.31


In [ ]:
# Comparaison avec APP-v1 sur VALIDATION (mêmes documents, même référence, même score).
docs_val = {d.document_id for d in parts["validation"]}
pred_v1 = {m for m in lire_predictions(RESULTATS / "predictions_apprise_validation.jsonl") if m.doc in docs_val}
comparaison = pd.concat([tableau_scores(ref_val, pred_v1, "apprise_v1", "validation"), scores_val], ignore_index=True)
comparaison.to_csv(SORTIE / "apprise_v2_comparaison_v1_validation.csv", index=False)

types_erreurs = pd.DataFrame({nom: {k: len(v) for k, v in erreurs(ref_val, p).items()}
                              for nom, p in [("APP-v1", pred_v1), ("APP-v2", pred_val)]}).T
display(comparaison)
types_erreurs

/tmp/ipykernel_1365/1429677279.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparaison = pd.concat([tableau_scores(ref_val, pred_v1, "apprise_v1", "validation"), scores_val], ignore_index=True)


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,apprise_v1,validation,Chemical,4886,473,461,0.9117,0.9138,0.9128,False,NaN
1,apprise_v1,validation,Disease,3322,938,922,0.7798,0.7828,0.7813,False,NaN
2,apprise_v1,validation,global (micro),8208,1411,1383,0.8533,0.8558,0.8546,False,NaN
3,apprise_v2,validation,Chemical,5018,360,329,0.9331,0.9385,0.9358,False,NaN
4,apprise_v2,validation,Disease,3588,719,656,0.8331,0.8454,0.8392,False,NaN
5,apprise_v2,validation,global (micro),8606,1079,985,0.8886,0.8973,0.8929,False,18.31


,correcte,frontiere,type,manquee,parasite
APP-v1,8208,631,11,741,769
APP-v2,8606,489,14,482,576


## Étape 4 — Prédictions sur TEST

Le modèle figé est appliqué aux textes de TEST ; les scores TEST seront calculés dans `5_evaluation.ipynb`,
une seule fois, si APP-v2 est retenue.

In [ ]:
t0 = time.perf_counter()
pred_test = predire(modele, tokenizer, fenetres["test"], bio_par_doc, PARAMS["max_length"])
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test, "apprise_v2", parts["test"], SORTIE / "predictions_apprise_v2_test.jsonl")

gel = {"version": VERSION, "modele": MODELE, "parametres": PARAMS, "taux_compares": TAUX_COMPARES, "graines": GRAINES,
       "config_retenue": {"learning_rate": float(choix.taux), "epochs": int(choix.epoques), "graine": GRAINE_FIGEE,
                          "F1_validation_moyen": float(choix.F1), "regle": f"meilleure moyenne, tolérance {TOLERANCE}, moins d'époques"},
       "politique_decodage": POLITIQUE_DECODAGE, "fenetres": "phrases entières, max_length sous-mots",
       "alignement_sous_mots": "premier sous-mot supervisé (tals.aligner_sous_mots)",
       "appareil": appareil, "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "temps_s": {"entrainement_10_epoques": float(point.temps_entrainement_s), "prediction_validation": round(t_val, 1),
                   "prediction_test": round(t_test, 1)},
       "fenetres_par_partition": {p: len(f) for p, f in fenetres.items()},
       "predictions_test": len(pred_test), "mode_rapide": MODE_RAPIDE}
(SORTIE / "apprise_v2_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gel, ensure_ascii=False, indent=2))